# 7장. RAG 시스템 평가 — 05. Heuristic 평가 (ROUGE, BLEU, METEOR, SemScore)

## 학습 목표

- **Heuristic(휴리스틱) 평가**의 개념과 장점 이해
- ROUGE, BLEU, METEOR, SemScore 네 가지 지표의 원리와 차이 파악
- LLM 호출 없이 빠르고 저렴하게 평가하는 방법 습득

## 사전 지식

- 04번 노트북에서 커스텀 평가자 작성 경험
- N-gram의 개념 (단어 연속열)

## Heuristic 평가란?

LLM 호출 없이 **통계적 알고리즘**으로 텍스트 품질을 측정하는 방법이에요.

| 특성 | Heuristic | LLM-as-Judge |
|------|-----------|--------------|
| **속도** | 밀리초 단위 | 1~3초 |
| **비용** | 없음 | API 비용 |
| **일관성** | 항상 동일 | 확률적 변동 |
| **정교함** | 제한적 | 높음 |
| **참조 답변** | 필수 | 선택적 |

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai rouge-score nltk sentence-transformers


In [ ]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'✅ 설정됨' if os.getenv('LANGCHAIN_API_KEY') else '❌ 미설정'}")
print(f"LANGCHAIN_TRACING_V2: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
print(f"LANGCHAIN_PROJECT: {os.getenv('LANGCHAIN_PROJECT', 'default')}")


---

## RAG 시스템 구축

앞 노트북과 동일한 RAG 시스템을 구축해요.

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 및 평가용 함수 정의
# ---------------------------------------------------
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 문서 로드 및 분할
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 2단계: 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """Answer the question based on the context provided. Be concise and specific.

Context: {context}

Question: {question}

Answer:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 평가용 함수: LangSmith evaluate()에 전달하는 래퍼 함수
def rag_answer_function(inputs: dict) -> dict:
    """RAG 시스템 답변 함수"""
    question = inputs["question"]
    answer = rag_chain.invoke(question)
    return {"answer": answer}

print("RAG 시스템 구축 완료")

---

## 1. ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**자동 요약 평가**에 가장 많이 사용되는 지표예요. 생성 텍스트와 참조 텍스트의 N-gram 중복을 측정하며, **Recall(재현율)** 중심으로 설계되었어요.

### ROUGE 종류

| 종류 | 측정 단위 | 특징 |
|------|---------|------|
| **ROUGE-1** | 단어(unigram) | 개별 단어 일치 |
| **ROUGE-2** | 2-gram | 연속 2단어 일치 |
| **ROUGE-L** | 최장 공통 부분 수열(LCS) | 순서를 고려한 일치 |

> **실무 팁**: RAG 시스템 평가에는 ROUGE-L이 가장 적합해요. 단어 순서도 고려하면서 유연한 비교가 가능합니다.

In [ ]:
# ---------------------------------------------------
# ROUGE 평가자 구현 (factory 패턴으로 metric 선택 가능)
# ---------------------------------------------------
# ============================================================
# TODO: rouge_evaluator(metric="rougeL") 팩토리 함수를 완성하세요
# 힌트: RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)로 점수 계산 후 fmeasure 반환
# 예상 결과: {"key": "ROUGE", "score": 0~1}
# ============================================================

from langsmith.schemas import Run, Example
from rouge_score import rouge_scorer



---

## 2. BLEU (Bilingual Evaluation Understudy)

원래 **기계 번역 평가**를 위해 개발된 지표로, **Precision(정밀도)** 중심이에요. 1~4-gram 정밀도의 기하평균으로 계산됩니다.

> **주의**: BLEU는 짧은 답변에 불리하고, 단어 순서보다 단어 선택에 치중해요. 번역이 아닌 QA 태스크에서는 ROUGE-L이 더 적합한 경우가 많습니다.

In [ ]:
# ---------------------------------------------------
# BLEU 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: sentence_bleu([reference_tokens], prediction_tokens)를 사용하는 평가자를 완성하세요
# 힌트: 공백으로 split()하여 토큰화하고, ZeroDivisionError 예외 처리 필요
# 예상 결과: {"key": "BLEU", "score": 0~1}
# ============================================================

from nltk.translate.bleu_score import sentence_bleu
import warnings
warnings.filterwarnings('ignore')



---

## 3. METEOR (Metric for Evaluation of Translation with Explicit ORdering)

BLEU보다 **유연한 평가**를 제공해요. 어간(stem)과 동의어(synonym)를 함께 고려해서 표면적으로 다른 단어도 의미가 같으면 일치로 처리합니다.

**예시**: "quick"와 "fast"는 BLEU에서 불일치지만 METEOR에서는 일치

In [ ]:
# ---------------------------------------------------
# METEOR 평가자 구현 (동의어, 어간 고려)
# ---------------------------------------------------
# ============================================================
# TODO: meteor_score.meteor_score([reference_tokens], prediction_tokens)를 사용하는 평가자를 완성하세요
# 힌트: WordNet 리소스를 먼저 다운로드하고 split()으로 토큰화
# 예상 결과: {"key": "METEOR", "score": 0~1}
# ============================================================

from nltk.translate import meteor_score
from nltk.corpus import wordnet as wn
import nltk

# WordNet 다운로드 (최초 1회) — 동의어 판단에 사용
try:
    wn.ensure_loaded()
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    wn.ensure_loaded()



---

## 4. SemScore (Semantic Similarity Score)

**문장 임베딩(Sentence Embedding)** 기반 의미 유사도 측정이에요. N-gram 기반 지표들과 달리 단어가 달라도 의미가 같으면 높은 점수를 줍니다.

**차이 예시**:

```
참조: "The cat is on the mat"
예측: "A feline is on the rug"

ROUGE-1: 낮음 (단어 일치 없음)
SemScore: 높음 (의미가 동일)
```

In [ ]:
# ---------------------------------------------------
# SemScore 평가자 구현 (문장 임베딩 기반)
# ---------------------------------------------------
# ============================================================
# TODO: SentenceTransformer를 사용해 의미 유사도 평가자를 완성하세요
# 힌트: sem_model.encode(text, convert_to_tensor=True), util.pytorch_cos_sim(emb1, emb2).item()
# 예상 결과: {"key": "sem_score", "score": 0~1}
# ============================================================

from sentence_transformers import SentenceTransformer, util
import warnings
warnings.filterwarnings('ignore')

# SentenceTransformer: 문장 수준 임베딩을 위한 경량 모델
# all-mpnet-base-v2: 의미 유사도 태스크에 최적화된 모델
sem_model = SentenceTransformer("all-mpnet-base-v2")



---

## LangSmith 데이터셋 준비

In [ ]:
from langsmith import Client
from langsmith import utils as ls_utils

client = Client()
dataset_name = "transformer-qa-heuristic-eval"

# 평가용 데이터 (질문 + 참조 답변)
qa_pairs = [
    {
        "question": "What is the Transformer architecture?",
        "answer": "The Transformer is a neural network architecture that relies entirely on attention mechanisms, without using recurrent or convolutional layers."
    },
    {
        "question": "How does self-attention work?",
        "answer": "Self-attention computes attention weights between all positions in a sequence, allowing the model to weigh the importance of different parts of the input."
    },
    {
        "question": "What are the advantages of Transformers?",
        "answer": "Transformers can process sequences in parallel, capture long-range dependencies more effectively, and are more efficient to train than RNNs."
    },
]

# 데이터셋 생성 또는 사용
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ 기존 데이터셋 사용: {dataset_name}")
# 구체적 예외 타입을 지정하면 예상치 못한 오류(네트워크 장애 등)를 놓치지 않습니다
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Transformer Q&A for heuristic evaluation"
    )
    
    for qa in qa_pairs:
        client.create_example(
            inputs={"question": qa["question"]},
            outputs={"answer": qa["answer"]},
            dataset_id=dataset.id
        )
    print(f"✅ 새 데이터셋 생성: {dataset_name} ({len(qa_pairs)}개 예제)")

---

## 4가지 Heuristic 평가 실행

네 가지 평가자를 한꺼번에 실행해서 결과를 비교해볼게요.

In [ ]:
# ---------------------------------------------------
# 4가지 Heuristic 평가자 조합하여 종합 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()에 4가지 heuristic 평가자를 전달하여 평가를 실행하세요
# 힌트: evaluators=[rouge_evaluator(metric="rougeL"), bleu_evaluator, meteor_evaluator, semscore_evaluator]
# 예상 결과: experiment_results.experiment_url 출력
# ============================================================

from langsmith.evaluation import evaluate



---

## 정리

### Heuristic 지표 비교

| 지표 | 측정 방식 | 장점 | 단점 | 적합한 태스크 |
|------|---------|------|------|-------------|
| **ROUGE** | N-gram 중복 (Recall) | 요약 평가 표준 | 표면적 일치만 측정 | 자동 요약 |
| **BLEU** | N-gram 정밀도 | 번역 평가 표준 | 짧은 문장에 불리 | 기계 번역 |
| **METEOR** | 동의어+어간+순서 | 유연한 평가 | 계산 복잡도 높음 | 번역, 요약 |
| **SemScore** | 임베딩 코사인 유사도 | 의미 포착 우수 | 참조 답변 필수 | QA, 요약 |

### N-gram 지표의 한계

N-gram 지표(ROUGE, BLEU, METEOR)는 표면적 단어 일치에 의존해요. 단어를 바꿔도 의미가 같은 경우를 놓칩니다. RAG 시스템의 답변은 표현 방식이 다양할 수 있으므로 **SemScore와 함께 사용하는 것**이 좋아요.

### 지표 선택 가이드

- **요약 품질 체크**: ROUGE
- **번역 품질 체크**: BLEU 또는 METEOR
- **의미 유사도가 중요할 때**: SemScore
- **종합 평가**: 네 가지 모두 + LLM-as-Judge

### 다음 노트북 예고

다음에는 RAG 시스템에서 가장 중요한 **Groundedness(근거성) 평가** — 답변이 컨텍스트에만 근거하는지, 즉 할루시네이션이 있는지 — 를 집중적으로 다뤄볼게요.